# 03 — Extração de tópicos — Ensemble vs BoW vs BERTopic

1. BOW -> contagem de palavras (é uma abordagem legal mas geralmente não consegue entender a semantica das palavras)
2. Ensemble - focado na abordagem de mitigar a quantidade de erros em cada modelo, dessa forma usei 3 modelos (1 baseado em semantica e 2 estatisticos que são mais rapidos para poder realizar a extração)
3. Bertopics (Cluster de Embeddings)

## 1. Setup

As instalações abaixo estão comentadas para evitar reinstalações desnecessárias. Descomente no Colab ou em ambiente novo.


In [ ]:
# Instalação opcional
# %pip install -q pandas numpy tqdm beautifulsoup4 scikit-learn nltk spacy
# %pip install -q keybert sentence-transformers yake rake-nltk bertopic
# %python -m spacy download pt_core_news_sm
# %pip install -q torch


Note: you may need to restart the kernel to use updated packages.
^C
Note: you may need to restart the kernel to use updated packages.


UsageError: Line magic function `%python` not found (But cell magic `%%python` exists, did you mean that instead?).


In [2]:
from pathlib import Path
import json
import re
import time
import warnings
import unicodedata
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 240)

c:\Users\joaov\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.1. Detecção de GPU/CPU

Esta célula tenta usar GPU para os modelos baseados em embeddings, especialmente KeyBERT e BERTopic. Se não houver CUDA disponível, o notebook segue em CPU.


In [3]:

def detect_device(prefer_gpu: bool = True):
    """Detecta CUDA/MPS/CPU sem quebrar o notebook quando torch não estiver instalado."""
    try:
        import torch
        if prefer_gpu and torch.cuda.is_available():
            device = 'cuda'
            device_name = torch.cuda.get_device_name(0)
        elif prefer_gpu and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            device = 'mps'
            device_name = 'Apple Silicon MPS'
        else:
            device = 'cpu'
            device_name = 'CPU'
        return torch, device, device_name
    except Exception as e:
        print('PyTorch não disponível ou falhou ao detectar acelerador. Usando CPU.')
        print(repr(e))
        return None, 'cpu', 'CPU'

TORCH, DEVICE, DEVICE_NAME = detect_device(prefer_gpu=True)
print(f'Dispositivo selecionado: {DEVICE} ({DEVICE_NAME})')


Dispositivo selecionado: cpu (CPU)


## 2. Caminhos de entrada e saída

Este notebook usa como base o arquivo limpo produzido pelo cleaning.


In [4]:
DATA_DIR = Path('../data')
OUTPUT_DIR = DATA_DIR / 'nip_pipeline_outputs'
PROCESSED_DIR = OUTPUT_DIR / 'processed'
TOPICS_DIR = OUTPUT_DIR / 'topics'
REPORTS_DIR = OUTPUT_DIR / 'reports'

TOPICS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CLEAN_FILE = PROCESSED_DIR / 'investimento_2016_2024_clean_step01.csv'

OUTPUT_TOPICS_NEWS = TOPICS_DIR / 'topics_comparison_step03.csv'
OUTPUT_TOPIC_MENTIONS = TOPICS_DIR / 'topic_mentions_step03.csv'
OUTPUT_BERTOPIC_INFO = TOPICS_DIR / 'bertopic_info_step03.csv'
OUTPUT_REPORT = REPORTS_DIR / 'summary_topics_step03.json'

INPUT_CLEAN_FILE

WindowsPath('../data/nip_pipeline_outputs/processed/investimento_2016_2024_clean_step01.csv')

## 3. Parâmetros

Ajuste `SAMPLE_SIZE` para testar com uma amostra menor antes de rodar no conjunto inteiro.


In [5]:
# Tamanho da amostra. Use None para rodar em todo o dataset.
SAMPLE_SIZE = 3000

# Número máximo de tópicos/palavras-chave por notícia por método.
TOP_N = 8

# Número mínimo de modelos que precisam concordar no ensemble.
MIN_VOTOS = 2

# Modelo de embeddings usado no KeyBERT e no BERTopic.
EMBED_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'

# Coluna textual produzida pelo cleaning.
TEXT_COLUMN = 'texto_base'

# Tamanho máximo do texto usado por documento. Ajuda custo/tempo em KeyBERT/BERTopic.
MAX_CHARS_PER_DOC = 6000

# Ative/desative métodos conforme o ambiente.
USE_BOW = True
USE_ENSEMBLE = True
USE_BERTOPIC = True

SEED = 42

## 4. Carregar base limpa


In [6]:
if not INPUT_CLEAN_FILE.exists():
    raise FileNotFoundError(
        f'Arquivo não encontrado: {INPUT_CLEAN_FILE}. Execute primeiro o notebook 01-clean-data.ipynb.'
    )

df = pd.read_csv(INPUT_CLEAN_FILE)
print(f'Linhas disponíveis: {len(df):,}')
print(f'Colunas: {len(df.columns):,}')
display(df.head(3))

Linhas disponíveis: 15,211
Colunas: 12


,noticia_id,data_publicacao,titulo,titulo_limpo,fonte,texto,texto_limpo,texto_base,flagnoticia,titulo_len,texto_len,texto_base_len
0,5c2f55d84fa33a4521c84952,2016-01-01 00:00:00,Araçoiaba busca melhorar abastecimento de água,Araçoiaba busca melhorar abastecimento de água,Diário de Sorocaba,"Para atender a demanda de crescimento dos bairros abastecidos pelo Reservatório Suíço, a concessionária Águas de Araçoiaba finalizou, neste mês de dezembro, melhorias no sistema de bombeamento da unidade. As obras tiveram início em outu...","Para atender a demanda de crescimento dos bairros abastecidos pelo Reservatório Suíço, a concessionária Águas de Araçoiaba finalizou, neste mês de dezembro, melhorias no sistema de bombeamento da unidade. As obras tiveram início em outu...","Araçoiaba busca melhorar abastecimento de água\nPara atender a demanda de crescimento dos bairros abastecidos pelo Reservatório Suíço, a concessionária Águas de Araçoiaba finalizou, neste mês de dezembro, melhorias no sistema de bombeam...",I,46,1580,1627
1,265e1fc4bd247c649173c83d,2016-01-02 00:00:00,Franquias da região conquistam mercado local e internacional,Franquias da região conquistam mercado local e internacional,Folha da Região,"Quem caminha por ruas comerciais e pelos centros de compras na região avista um grande número de fachadas de franquias, a maioria importada de outras partes do Brasil. Mas há redes de franchising que fazem o caminho inverso. Algumas emp...","Quem caminha por ruas comerciais e pelos centros de compras na região avista um grande número de fachadas de franquias, a maioria importada de outras partes do Brasil. Mas há redes de franchising que fazem o caminho inverso. Algumas emp...","Franquias da região conquistam mercado local e internacional\nQuem caminha por ruas comerciais e pelos centros de compras na região avista um grande número de fachadas de franquias, a maioria importada de outras partes do Brasil. Mas há...",I,60,3027,3088
2,72519e6106b0f258b1880485,2016-01-02 00:00:00,"Hospital da Criança, do Grendac, será entregue até março deste ano","Hospital da Criança, do Grendac, será entregue até março deste ano",Jornal de Jundiaí,"Obras do Hospital da Criança, que vai atender diversas especialidades médicas além da oncologia, estão dentro do cronograma\nA expectativa dos administradores do Grupo em Defesa da Criança com Câncer (Grendacc), em Jundiaí, é de que o H...","Obras do Hospital da Criança, que vai atender diversas especialidades médicas além da oncologia, estão dentro do cronograma\nA expectativa dos administradores do Grupo em Defesa da Criança com Câncer (Grendacc), em Jundiaí, é de que o H...","Hospital da Criança, do Grendac, será entregue até março deste ano\nObras do Hospital da Criança, que vai atender diversas especialidades médicas além da oncologia, estão dentro do cronograma\nA expectativa dos administradores do Grupo ...",I,66,2384,2451


## 5. Validação mínima da entrada

O arquivo limpo deve ter `noticia_id`, `texto_base`, `titulo`, `fonte`, `data_publicacao` e, se disponível, `flagnoticia`.


In [7]:
required_cols = {'noticia_id', TEXT_COLUMN}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Colunas obrigatórias ausentes no arquivo limpo: {missing}')

if 'flagnoticia' in df.columns:
    df['flagnoticia'] = df['flagnoticia'].astype('string').str.strip().str.upper()
    before = len(df)
    df = df[df['flagnoticia'].eq('I')].copy()
    print(f'Filtro flagnoticia=I: {before:,} → {len(df):,}')
else:
    print('Coluna flagnoticia não encontrada; usando todas as linhas do arquivo limpo.')

# Garante texto não vazio.
df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna('').astype(str)
df = df[df[TEXT_COLUMN].str.strip().ne('')].copy()
print(f'Linhas com texto: {len(df):,}')

display(df[['noticia_id', 'data_publicacao', 'fonte', 'titulo']].head(5))

Filtro flagnoticia=I: 15,211 → 15,211
Linhas com texto: 15,211


,noticia_id,data_publicacao,fonte,titulo
0,5c2f55d84fa33a4521c84952,2016-01-01 00:00:00,Diário de Sorocaba,Araçoiaba busca melhorar abastecimento de água
1,265e1fc4bd247c649173c83d,2016-01-02 00:00:00,Folha da Região,Franquias da região conquistam mercado local e internacional
2,72519e6106b0f258b1880485,2016-01-02 00:00:00,Jornal de Jundiaí,"Hospital da Criança, do Grendac, será entregue até março deste ano"
3,729d407367d6ce010c660b27,2016-01-02 00:00:00,Jornal de Jundiaí,"Hospital da Criança,do Grendacc,será entregue até março deste ano"
4,eb94b4766db90094be9dc437,2016-01-02 00:00:00,Cruzeiro do Sul,Produtor investe R$ 100 mil em estufa


## 6. Preparação textual para tópicos

Aqui fazemos uma preparação **específica para tópicos**, sem alterar o arquivo limpo original. Diferentemente da NER e de eventos, a extração de tópicos pode se beneficiar de uma versão mais plana do texto.


In [8]:
def clean_text_for_topics(value: str, max_chars: int | None = None) -> str:
    if pd.isna(value):
        return ''
    text = str(value)
    text = BeautifulSoup(text, 'html.parser').get_text(' ')
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'[\x00-\x1f\x7f]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if max_chars is not None and len(text) > max_chars:
        text = text[:max_chars]
    return text

# Alguns padrões de ruído jornalístico/editorial que podem atrapalhar tópicos.
LIXO_TITULO = [
    r'página\s+impress',
    r'^\s*fonte\s*:',
    r'\(\d{2}/\d{2}/\d{4}\)\s*-\s*fonte',
    r'^p[áa]g(ina)?\s*\d+',
]

def is_noise_title(title: str) -> bool:
    t = str(title).lower()
    return any(re.search(pattern, t) for pattern in LIXO_TITULO)

df_work = df.copy()
if 'titulo' in df_work.columns:
    before = len(df_work)
    df_work = df_work[~df_work['titulo'].apply(is_noise_title)].copy()
    print(f'Títulos possivelmente ruidosos removidos: {before - len(df_work):,}')

df_work['texto_para_topicos'] = df_work[TEXT_COLUMN].apply(lambda x: clean_text_for_topics(x, MAX_CHARS_PER_DOC))
df_work['n_words_topics'] = df_work['texto_para_topicos'].str.split().str.len()
df_work = df_work[df_work['n_words_topics'] >= 20].copy()

if SAMPLE_SIZE is not None:
    df_work = df_work.sample(min(SAMPLE_SIZE, len(df_work)), random_state=SEED).reset_index(drop=True)
else:
    df_work = df_work.reset_index(drop=True)

textos = df_work['texto_para_topicos'].tolist()
print(f'Notícias selecionadas para extração de tópicos: {len(df_work):,}')
display(df_work[['noticia_id', 'titulo', 'n_words_topics']].head(5))

Títulos possivelmente ruidosos removidos: 357
Notícias selecionadas para extração de tópicos: 3,000


,noticia_id,titulo,n_words_topics
0,253c1cbb44afc7e223893b25,Rafa Prado fala sobre empreendedorismo no Diálogo Diário,248
1,5b6c859eb8607abb43d426fe,Mercado Livre quer ser um Alibaba,980
2,edfcb3229831f96faaea0194,Artista plástico expõe obras em S.Caetano,334
3,aa5034e77790c8130dae8d2a,"Com produtos 100% vegetais, startup NotCo abre restaurante delivery em SP",536
4,43cf9819ef87797ce85b0f81,BDO planeja duas aquisições no Brasil,764


## 7. Stopwords e filtro nominal

Este filtro é usado somente para tópicos/palavras-chave. Ele não deve ser aplicado à extração de eventos, porque verbos são úteis para identificar ações de investimento.


In [9]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

STOP = set(stopwords.words('portuguese'))
STOP.update([
    'hoje', 'ontem', 'agora', 'aqui', 'ano', 'anos', 'dia', 'dias',
    'reais', 'r$', 'ser', 'vai', 'após', 'sobre', 'disse', 'afirmou',
    'segundo', 'milhões', 'milhão', 'mil', 'bilhões', 'bilhão',
    'página', 'impresso', 'fonte', 'cidade', 'geral', 'paulo', 'brasil',
])
STOP = sorted(STOP)
print(f'Stopwords: {len(STOP):,}')

ModuleNotFoundError: No module named 'nltk'

In [ ]:
# spaCy é usado apenas para filtrar candidatos nominais.
# Se o modelo pt_core_news_sm não estiver instalado, o notebook segue sem filtro POS.
try:
    import spacy
    nlp = spacy.load('pt_core_news_sm', exclude=['parser', 'lemmatizer', 'textcat'])
    HAS_SPACY = True
    print('spaCy carregado: pt_core_news_sm')
except Exception as e:
    nlp = None
    HAS_SPACY = False
    print('spaCy indisponível. Filtro nominal será simplificado.')
    print(e)

_pos_cache = {}

def is_nominal_phrase(phrase: str) -> bool:
    phrase = str(phrase).strip().lower()
    if not phrase or len(phrase) < 3:
        return False
    if phrase in _pos_cache:
        return _pos_cache[phrase]
    if not HAS_SPACY:
        # Fallback simples: remove frases numéricas e termos de uma letra.
        tokens = re.findall(r'[a-zA-ZÀ-ÿ]{3,}', phrase)
        ok = len(tokens) > 0
        _pos_cache[phrase] = ok
        return ok
    doc = nlp(phrase)
    tags = [token.pos_ for token in doc]
    ok = bool(tags) and 'VERB' not in tags and 'AUX' not in tags and any(tag in {'NOUN', 'PROPN'} for tag in tags)
    _pos_cache[phrase] = ok
    return ok

for candidate in ['produtor investe', 'produtor rural', 'energia solar', 'obras começaram ontem']:
    print(f'{candidate:25s} -> {is_nominal_phrase(candidate)}')

## 8. Método 1 — BoW/TF-IDF por documento

O BoW/TF-IDF funciona como baseline estatístico. Ele tende a ser rápido, mas pode capturar termos menos semânticos que os métodos baseados em embeddings.


In [ ]:
def normalize_topic_text(value: str) -> str:
    value = str(value).lower().strip()
    value = unicodedata.normalize('NFKC', value)
    value = re.sub(r'\s+', ' ', value)
    return value

if USE_BOW:
    from sklearn.feature_extraction.text import TfidfVectorizer

    tfidf = TfidfVectorizer(
        stop_words=STOP,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.6,
    )
    M = tfidf.fit_transform(textos).tocsr()
    vocab = np.array(tfidf.get_feature_names_out())

    def extract_bow_keywords(i: int) -> list[str]:
        row = M.getrow(i)
        if row.nnz == 0:
            return []
        ordered = row.indices[np.argsort(row.data)[::-1]]
        candidates = [normalize_topic_text(vocab[idx]) for idx in ordered[: TOP_N * 4]]
        output = []
        for candidate in candidates:
            if is_nominal_phrase(candidate) and candidate not in output:
                output.append(candidate)
            if len(output) >= TOP_N:
                break
        return output

    df_work['topics_bow'] = [extract_bow_keywords(i) for i in range(len(df_work))]
    print('BoW/TF-IDF concluído.')
else:
    df_work['topics_bow'] = [[] for _ in range(len(df_work))]
    print('BoW/TF-IDF desativado.')

## 9. Método 2 — Ensemble de YAKE, RAKE e KeyBERT

O ensemble combina dois métodos estatísticos/heurísticos com um método baseado em embeddings. A concordância entre métodos é usada como sinal de estabilidade do tópico.


In [ ]:

if USE_ENSEMBLE:
    import yake
    from rake_nltk import Rake
    from keybert import KeyBERT
    from sentence_transformers import SentenceTransformer

    print('Device para embeddings:', DEVICE)

    yake_extractor = yake.KeywordExtractor(lan='pt', n=3, top=TOP_N * 3, dedupLim=0.7)

    def extract_yake(text: str) -> list[str]:
        try:
            values = sorted(yake_extractor.extract_keywords(text), key=lambda x: x[1])
            return [normalize_topic_text(keyword) for keyword, score in values]
        except Exception:
            return []

    rake_extractor = Rake(
        stopwords=set(STOP),
        max_length=3,
        sentence_tokenizer=lambda x: re.split(r'(?<=[.!?])\s+', x),
        word_tokenizer=lambda x: re.findall(r'[a-zA-ZÀ-ÿ]+', x),
    )

    def extract_rake(text: str) -> list[str]:
        try:
            rake_extractor.extract_keywords_from_text(text)
            return [normalize_topic_text(p) for p in rake_extractor.get_ranked_phrases()[: TOP_N * 3]]
        except Exception:
            return []

    try:
        sentence_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
    except Exception as e:
        print(f'Falha ao carregar SentenceTransformer em {DEVICE}. Tentando CPU.')
        print(repr(e))
        DEVICE = 'cpu'
        sentence_model = SentenceTransformer(EMBED_MODEL, device='cpu')

    keybert_model = KeyBERT(model=sentence_model)
    print(f'KeyBERT/SentenceTransformer executando em: {DEVICE}')

    def extract_keybert_batch(texts: list[str]) -> list[list[str]]:
        try:
            results = keybert_model.extract_keywords(
                texts,
                keyphrase_ngram_range=(1, 3),
                stop_words=STOP,
                use_mmr=True,
                diversity=0.6,
                top_n=TOP_N * 3,
            )
            # KeyBERT retorna lista de tuplas para texto único e lista de listas para batch.
            if results and isinstance(results[0], tuple):
                results = [results]
            return [[normalize_topic_text(keyword) for keyword, score in item] for item in results]
        except Exception as e:
            print('Falha no KeyBERT:', e)
            return [[] for _ in texts]

else:
    print('Ensemble desativado.')


In [ ]:
if USE_ENSEMBLE:
    t0 = time.time()
    topics_yake = [extract_yake(text) for text in tqdm(textos, desc='YAKE')]
    print('YAKE:', round(time.time() - t0, 1), 's')

    t0 = time.time()
    topics_rake = [extract_rake(text) for text in tqdm(textos, desc='RAKE')]
    print('RAKE:', round(time.time() - t0, 1), 's')

    t0 = time.time()
    topics_keybert = extract_keybert_batch(textos)
    print('KeyBERT:', round(time.time() - t0, 1), 's')
else:
    topics_yake = [[] for _ in range(len(df_work))]
    topics_rake = [[] for _ in range(len(df_work))]
    topics_keybert = [[] for _ in range(len(df_work))]

df_work['topics_yake'] = topics_yake
df_work['topics_rake'] = topics_rake
df_work['topics_keybert'] = topics_keybert

In [ ]:
def consensus_topics(topic_lists: list[list[str]], min_votes: int = MIN_VOTOS, top_n: int = TOP_N) -> list[str]:
    votes = defaultdict(set)
    for method_idx, topics in enumerate(topic_lists):
        for topic in topics:
            topic = normalize_topic_text(topic)
            if len(topic) < 3:
                continue
            votes[topic].add(method_idx)

    # Propaga votos entre expressões muito próximas por inclusão de tokens.
    token_sets = {topic: set(topic.split()) for topic in votes}
    topics = list(votes.keys())
    for a in topics:
        for b in topics:
            if a == b:
                continue
            if not token_sets[a] or not token_sets[b]:
                continue
            if token_sets[a] <= token_sets[b] or token_sets[b] <= token_sets[a]:
                votes[a] |= votes[b]

    candidates = sorted(
        [(topic, len(methods)) for topic, methods in votes.items() if len(methods) >= min_votes],
        key=lambda x: (-x[1], x[0]),
    )

    output = []
    for topic, score in candidates:
        if not is_nominal_phrase(topic):
            continue
        if any(topic != existing and (topic in existing or existing in topic) for existing in output):
            continue
        output.append(topic)
        if len(output) >= top_n:
            break
    return output

if USE_ENSEMBLE:
    df_work['topics_ensemble'] = [
        consensus_topics([yake_topics, rake_topics, keybert_topics])
        for yake_topics, rake_topics, keybert_topics in zip(topics_yake, topics_rake, topics_keybert)
    ]
else:
    df_work['topics_ensemble'] = [[] for _ in range(len(df_work))]

print('Consenso ensemble concluído.')

## 10. Método 3 — BERTopic

O BERTopic é mais global: ele agrupa documentos em clusters semânticos e depois representa cada cluster por termos relevantes.


In [ ]:

if USE_BERTOPIC:
    from bertopic import BERTopic
    from sklearn.feature_extraction.text import CountVectorizer
    from sentence_transformers import SentenceTransformer

    vectorizer_model = CountVectorizer(stop_words=STOP, ngram_range=(1, 2))

    try:
        # Reutiliza o sentence_model do KeyBERT quando ele já existir.
        bertopic_embedding_model = sentence_model if 'sentence_model' in globals() else SentenceTransformer(EMBED_MODEL, device=DEVICE)
    except Exception as e:
        print(f'Falha ao carregar embedding_model do BERTopic em {DEVICE}. Tentando CPU.')
        print(repr(e))
        DEVICE = 'cpu'
        bertopic_embedding_model = SentenceTransformer(EMBED_MODEL, device='cpu')

    topic_model = BERTopic(
        embedding_model=bertopic_embedding_model,
        vectorizer_model=vectorizer_model,
        language='multilingual',
        min_topic_size=10,
        calculate_probabilities=False,
        verbose=False,
    )

    print(f'BERTopic usando embeddings em: {DEVICE}')
    topics_ids, _ = topic_model.fit_transform(textos)

    try:
        topics_ids = topic_model.reduce_outliers(textos, topics_ids, strategy='embeddings')
    except Exception as e:
        print('reduce_outliers pulado:', e)

    def get_bertopic_keywords(topic_id: int) -> list[str]:
        if topic_id == -1:
            return []
        try:
            topic_terms = topic_model.get_topic(topic_id) or []
            return [normalize_topic_text(term) for term, score in topic_terms[:TOP_N] if is_nominal_phrase(term)]
        except Exception:
            return []

    df_work['bertopic_topic_id'] = topics_ids
    df_work['topics_bertopic'] = [get_bertopic_keywords(topic_id) for topic_id in topics_ids]

    try:
        topic_info = topic_model.get_topic_info()
        topic_info.to_csv(OUTPUT_BERTOPIC_INFO, index=False, encoding='utf-8')
    except Exception as e:
        print('Não foi possível salvar topic_info:', e)

    n_topics = len(set(topics_ids)) - (1 if -1 in set(topics_ids) else 0)
    print(f'BERTopic concluído: {n_topics} tópicos, {sum(t == -1 for t in topics_ids)} outliers.')
else:
    df_work['bertopic_topic_id'] = -1
    df_work['topics_bertopic'] = [[] for _ in range(len(df_work))]
    print('BERTopic desativado.')


## 11. Definir tópico final e comparar métodos

Por padrão, `topics` recebe o ensemble quando disponível. Se o ensemble estiver vazio, usa BERTopic; se ainda estiver vazio, usa BoW.


In [ ]:
def choose_final_topics(row) -> list[str]:
    for col in ['topics_ensemble', 'topics_bertopic', 'topics_bow']:
        values = row.get(col, [])
        if isinstance(values, list) and len(values) > 0:
            return values[:TOP_N]
    return []

df_work['topics'] = df_work.apply(choose_final_topics, axis=1)

comparison_cols = [
    'noticia_id', 'data_publicacao', 'fonte', 'titulo',
    'topics', 'topics_bow', 'topics_yake', 'topics_rake', 'topics_keybert',
    'topics_ensemble', 'topics_bertopic', 'bertopic_topic_id',
]
comparison_cols = [col for col in comparison_cols if col in df_work.columns]

display(df_work[comparison_cols].head(10))

## 12. Criar tabela longa de menções de tópicos

Essa tabela é mais adequada para o Neo4j do que listas dentro de uma coluna, porque cada linha representa uma relação potencial `Noticia -> Topico`.


In [ ]:
topic_method_columns = {
    'bow': 'topics_bow',
    'yake': 'topics_yake',
    'rake': 'topics_rake',
    'keybert': 'topics_keybert',
    'ensemble': 'topics_ensemble',
    'bertopic': 'topics_bertopic',
    'final': 'topics',
}

mention_rows = []
for _, row in df_work.iterrows():
    for method, col in topic_method_columns.items():
        topics_list = row.get(col, [])
        if not isinstance(topics_list, list):
            continue
        for rank, topic in enumerate(topics_list, start=1):
            topic_norm = normalize_topic_text(topic)
            if not topic_norm:
                continue
            mention_rows.append({
                'noticia_id': row['noticia_id'],
                'method': method,
                'topic_text': topic,
                'topic_text_norm': topic_norm,
                'rank': rank,
                'bertopic_topic_id': row.get('bertopic_topic_id', None),
            })

topic_mentions = pd.DataFrame(mention_rows)
print(f'Menções de tópicos: {len(topic_mentions):,}')
display(topic_mentions.head(20))

## 13. Salvar saídas


In [ ]:
def list_to_string(values) -> str:
    if isinstance(values, list):
        return '; '.join(str(v) for v in values)
    if pd.isna(values):
        return ''
    return str(values)

out = df_work.copy()
for col in ['topics', 'topics_bow', 'topics_yake', 'topics_rake', 'topics_keybert', 'topics_ensemble', 'topics_bertopic']:
    if col in out.columns:
        out[col] = out[col].apply(list_to_string)

save_cols = [
    'noticia_id', 'data_publicacao', 'titulo', 'titulo_limpo', 'fonte',
    'texto_base', 'flagnoticia', 'topics', 'topics_bow', 'topics_yake',
    'topics_rake', 'topics_keybert', 'topics_ensemble', 'topics_bertopic',
    'bertopic_topic_id', 'n_words_topics',
]
save_cols = [col for col in save_cols if col in out.columns]

out[save_cols].to_csv(OUTPUT_TOPICS_NEWS, index=False, encoding='utf-8')
topic_mentions.to_csv(OUTPUT_TOPIC_MENTIONS, index=False, encoding='utf-8')

summary = {
    'created_at': datetime.now().isoformat(),
    'input_clean_file': str(INPUT_CLEAN_FILE),
    'n_news_available': int(len(df)),
    'n_news_processed': int(len(df_work)),
    'sample_size': SAMPLE_SIZE,
    'top_n': TOP_N,
    'min_votes': MIN_VOTOS,
    'embed_model': EMBED_MODEL,
    'device': DEVICE,
    'device_name': DEVICE_NAME,
    'methods_enabled': {
        'bow': bool(USE_BOW),
        'ensemble': bool(USE_ENSEMBLE),
        'bertopic': bool(USE_BERTOPIC),
    },
    'n_topic_mentions': int(len(topic_mentions)),
    'n_unique_final_topics': int(topic_mentions.loc[topic_mentions['method'].eq('final'), 'topic_text_norm'].nunique()) if not topic_mentions.empty else 0,
    'output_topics_news': str(OUTPUT_TOPICS_NEWS),
    'output_topic_mentions': str(OUTPUT_TOPIC_MENTIONS),
    'output_bertopic_info': str(OUTPUT_BERTOPIC_INFO),
}

with open(OUTPUT_REPORT, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'Arquivo por notícia salvo em: {OUTPUT_TOPICS_NEWS}')
print(f'Menções de tópicos salvas em: {OUTPUT_TOPIC_MENTIONS}')
print(f'Relatório salvo em: {OUTPUT_REPORT}')
display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))

## 14. Inspeção rápida dos resultados


In [ ]:
if not topic_mentions.empty:
    print('Top 30 tópicos finais por frequência:')
    display(
        topic_mentions[topic_mentions['method'].eq('final')]
        .groupby('topic_text_norm')
        .size()
        .sort_values(ascending=False)
        .head(30)
        .reset_index(name='freq')
    )

for i in range(min(10, len(df_work))):
    row = df_work.iloc[i]
    print(f"[{i}] {row.get('titulo', '')[:90]}")
    print('  BoW      :', ', '.join(row.get('topics_bow', [])))
    print('  YAKE     :', ', '.join(row.get('topics_yake', [])[:TOP_N]))
    print('  RAKE     :', ', '.join(row.get('topics_rake', [])[:TOP_N]))
    print('  KeyBERT  :', ', '.join(row.get('topics_keybert', [])[:TOP_N]))
    print('  Ensemble :', ', '.join(row.get('topics_ensemble', [])))
    print('  BERTopic :', ', '.join(row.get('topics_bertopic', [])))
    print('  Final    :', ', '.join(row.get('topics', [])))
    print()